#### Problem 9

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/metabric.csv")

survival_months = df["Overall Survival (Months)"].dropna().to_numpy()
original_mean = np.mean(survival_months)

In [4]:
seed = 100
n_bootstrap = 2000
rng = np.random.default_rng(seed)

bootstrap_means = []

for _ in range(n_bootstrap):
    bootstrap_sample = rng.choice(
        survival_months,
        size=len(survival_months),
        replace=True
    )

    bootstrap_means.append(np.mean(bootstrap_sample))

bootstrap_means = np.array(bootstrap_means)

print(f"Original mean: {original_mean:.4f} months")
print(f"Bootstrap mean: {np.mean(bootstrap_means):.4f} months")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {n_bootstrap}")


Original mean: 129.0943 months
Bootstrap mean: 129.0916 months
Seed: 100
Bootstrap replications: 2000


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the 1/(N −1) degrees of freedom correction).

In [6]:
bootstrap_se = np.std(bootstrap_means, ddof=1)

print(f"Bootstrap SE: {bootstrap_se:.4f} months")


Bootstrap SE: 2.0890 months


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [8]:
critical_values = {
    0.90: 1.645,
    0.95: 1.96,
    0.99: 2.576
}

confidence_intervals = {}

for confidence, critical_value in critical_values.items():
    margin_error = critical_value * bootstrap_se

    lower = original_mean - margin_error
    upper = original_mean + margin_error

    confidence_intervals[confidence] = (lower, upper)

    print(f"{confidence:.0%} CI: [{lower:.4f}, {upper:.4f}]")


90% CI: [125.6579, 132.5307]
95% CI: [124.9998, 133.1887]
99% CI: [123.7130, 134.4756]


d. Use each confidence interval to evaluate $H_0: M = 120$. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis.

In [9]:
null_mean = 120

for confidence, (lower, upper) in confidence_intervals.items():
    alpha = 1 - confidence

    if lower <= null_mean <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"Alpha = {alpha:.2f}: {decision}")


Alpha = 0.10: Reject H0
Alpha = 0.05: Reject H0
Alpha = 0.01: Reject H0


e. Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.

We have enough evidence to say that there is a statistically significant difference, meaning the mean recorded survival time differs from 120 months (10 years). We cannot estimate the complete lifetime of patients because we don't know when living patients will eventually die. All we know is that they were still alive.